# 01 · La foto pequeña de pared — Thomas y Nat

**Objetivo:** sincronizar los sprites `web/assets/props/room_photo.png` y `room_photo_torn.png` con la nueva seed 1888 ya integrada en los close-ups. **No requiere GPU**. V2: preserva exactamente el canal alfa RGBA original (esquinas y bordes), corrigiendo el defecto de la primera generación., y no consume las unidades GPU de Google AI Pro. 

Se conservará el marco original de 70×92, la zona de encuadre, el pie impreso y los píxeles exteriores. Solo se cambia la lámina fotográfica interior. Los dos PNG se generan a partir de las dos versiones 1888, con el mismo recorte y sin dibujar arañazos adicionales. Se exportan 3 intensidades de mezcla para comparar visualmente.

**No hace push ni sustituye assets reales.** Hay que revisar en capítulo 1 y capítulo 3 el sprite colgado, la sombra, las anclas de la cara del mayor y la coherencia con la fotografía ampliada.

In [ ]:
%pip -q install 'Pillow>=10' 'numpy>=1.24' 'requests>=2.31'

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance, ImageOps, ImageChops
from IPython.display import display
import base64, io, json, math, hashlib, random, requests, shutil
REPO = "gabsvm/HorrorPlay"
COMMIT = "0681cfee36921ec2bd372f8ae8080a401f8bd63b"

def source(path):
    url = f"https://api.github.com/repos/{REPO}/contents/{path}"
    r = requests.get(url, params={"ref": COMMIT}, headers={"Accept":"application/vnd.github+json"}, timeout=90)
    r.raise_for_status()
    data = r.json()
    if data.get("encoding") != "base64":
        raise RuntimeError(f"GitHub no devolvió el JPG/PNG en base64: {path}")
    b = base64.b64decode(data["content"])
    result = Image.open(io.BytesIO(b)).convert("RGB" if path.lower().endswith((".jpg", ".jpeg")) else "RGBA")
    return result, hashlib.sha256(b).hexdigest()

def font(size=20, bold=False):
    files = ["/usr/share/fonts/truetype/dejavu/DejaVuSerif-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSerif.ttf",
             "/usr/share/fonts/truetype/liberation2/LiberationSerif-Regular.ttf"]
    for f in files:
        if Path(f).exists(): return ImageFont.truetype(f, size)
    return ImageFont.load_default()

def compare(images, thumb=(640,360), labels=None):
    w,h=thumb
    sheet=Image.new('RGB',(w*len(images),h+36),(17,18,20))
    dr=ImageDraw.Draw(sheet)
    for i,im in enumerate(images):
        sheet.paste(im.convert('RGB').resize((w,h), Image.Resampling.LANCZOS),(i*w,36))
        dr.text((i*w+12,10), labels[i] if labels else str(i),fill=(245,241,229),font=font(15))
    return sheet

def finish(folder, manifest, outputs):
    manifest.update({"repo":REPO,"source_commit":COMMIT,"status":"CANDIDATO - No integrado ni testeado en Android"})
    (folder/'manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
    archive=shutil.make_archive(str(folder),'zip',folder)
    print('Exportado:', archive, 'archivos:', len(list(folder.iterdir())))
    from google.colab import files
    files.download(archive)

In [ ]:
from PIL import ImageStat
out=Path('/content/horrorplay_wall_candidates');out.mkdir(parents=True,exist_ok=True)
CURRENT={};CLOSEUP={};SHA={}
for variant,wp,cp in [
    ('intacta','web/assets/props/room_photo.png','web/assets/closeups/room_photo.jpg'),
    ('raspada','web/assets/props/room_photo_torn.png','web/assets/closeups/room_photo_torn.jpg')]:
    CURRENT[variant],SHA[wp]=source(wp)
    CLOSEUP[variant],SHA[cp]=source(cp)
assert all(im.size==(70,92) for im in CURRENT.values())
assert all(im.size==(1280,720) for im in CLOSEUP.values())
print('Fuentes y hashes:',SHA)
display(compare([CURRENT['intacta'],CURRENT['raspada']],(280,368),['Sprite original C1','Sprite original C3']))

## Recorte de la fotografía / máscara del interior

El rectángulo del original corresponde al cristal **dentro** del marco. La figura mayor debe mantenerse a la izquierda y Nat a la derecha. La composición del close-up es una *carte-de-visite* (x445–835, y83–575); al montarla al marco preservaremos solo su área interior. **Revisa este preview antes de continuar**: un desplazamiento de 1 píxel se nota mucho en el sprite 70×92.

In [ ]:
# Coordenadas calibrables; no alterar sin inspeccionar el preview.
SRC_CROP=(446,83,839,578)    # interior fotográfico del primer plano 1280×720
DST_BOX=(8,7,62,73)         # interior del sprite 70×92 (no tocar marco exterior)
assert DST_BOX[2]-DST_BOX[0]==54 and DST_BOX[3]-DST_BOX[1]==66

def candidate(kind, strength):
    base=CURRENT[kind].copy().convert('RGBA')
    source_alpha=base.getchannel('A').copy()
    interior=CLOSEUP[kind].crop(SRC_CROP).resize((54,66),Image.Resampling.LANCZOS)
    # Ajustar ligeramente la exposición del close-up para el cuarto oscuro.
    interior=ImageEnhance.Contrast(interior).enhance(.94)
    interior=ImageEnhance.Brightness(interior).enhance(.94)
    old=base.crop(DST_BOX).convert('RGB')
    mixed=Image.blend(old,interior,strength)
    # Mantener 2 px de transición al borde sin cambiar marco exterior.
    alpha=Image.new('L',(54,66),0);dr=ImageDraw.Draw(alpha)
    dr.rectangle((1,1,52,64),fill=255)
    alpha=alpha.filter(ImageFilter.GaussianBlur(.9))
    base.paste(Image.composite(mixed,old,alpha),DST_BOX)
    base.putalpha(source_alpha)  # CRÍTICO: esquinas, borde y zonas semitransparentes intactos
    assert ImageChops.difference(base.getchannel('A'),source_alpha).getbbox() is None
    return base

for k in ('intacta','raspada'):
    samples=[CURRENT[k].convert('RGB')]+[candidate(k,x) for x in (.70,.85,1.)]
    display(compare(samples,(280,368),[f'{k} actual','70%','85%','100%']))

## Exportación

Se exportan pares de PNG con nombres separados: ninguno sobreescribe lo que usa el juego. Seleccionar una sola intensidad y usarla tanto en la intacta como en la raspada. Control posterior: dibujarlas dentro del motor, no evaluar únicamente el PNG ampliado.

In [ ]:
from PIL import ImageChops
for strength in (.70,.85,1.):
    tag=str(int(round(strength*100)))
    for kind,filename in [('intacta','room_photo'),('raspada','room_photo_torn')]:
        im=candidate(kind,strength)
        original=CURRENT[kind].convert('RGBA')
        # Invariante: fuera del interior no cambia absolutamente ningún píxel.
        diff=ImageChops.difference(im.convert('RGB'),original.convert('RGB'))
        mask=Image.new('L',(70,92),255);ImageDraw.Draw(mask).rectangle(DST_BOX,fill=0)
        assert ImageChops.multiply(diff.convert('L'),mask).getbbox() is None
        assert ImageChops.difference(im.getchannel('A'),original.getchannel('A')).getbbox() is None, 'Alpha fue alterado'
        assert im.mode=='RGBA' and im.getpixel((0,0))[3] == original.getpixel((0,0))[3]
        im.save(out/f'{filename}_wall_{tag}.png')
manifest={'source_hashes':SHA,'src_crop':SRC_CROP,'dst_box':DST_BOX,
          'strengths':[70,85,100], 'requires':'Comparación visual del motor C1/C3 y hot spots'}
finish(out,manifest,[])